**[Course Home](../README.md) | [Python Lab](../python/09_jackson_networks_and_bottleneck_routing.ipynb) | [Prev: Ch 08](08_priority_queues_and_scheduling_disciplines.ipynb) | [Next: Appendix 10](10_appendix_the_queueing_theory_rosetta_stone.ipynb)**

*Part III: Scheduling, Networks & Global Fleet Optimization — Chapter 09 of 11*

---

# Chapter 9: Networks of Queues — Burke's Theorem, Jackson Networks & Bottleneck Routing
### *Multi-Stage Pipelines, Traffic Equations, Product-Form Equilibrium, and the Power-of-Two-Choices Law*

---

## 1. Beyond the Isolated Queue: Systems as Graphs

In preceding chapters, we analyzed queues in isolation: jobs arrive from an external world, wait in a buffer, receive service, and vanish into the ether.
In real-world software infrastructure, isolated queues are the exception. Production systems are directed graphs of interconnected queues:
- **Continuous Integration & Delivery Pipelines**: Pull requests flow through a multi-stage dependency graph: Linting $\to$ Compilation $\to$ Unit Testing $\to$ Integration Testing $\to$ Container Packaging.
- **Microservice RPC Graphs**: An incoming user request spawns sub-queries to an authentication service, a database pool, a caching tier, and an ML inference worker pool.
- **Feedback Loops**: When a test fails intermittently or a build cache entry is invalidated, jobs loop back to earlier stages for re-compilation or diagnostic re-runs.

How do we compute the capacity, bottleneck station, and end-to-end latency of a multi-stage queueing network with feedback loops?
This chapter develops the foundational theorems of queueing networks: Paul Burke's Output Theorem, James R. Jackson's Product-Form Law, bottleneck capacity allocation, and the revolutionary Power-of-Two-Choices load balancing principle.

---

## 2. Burke's Output Theorem: The Memoryless Cascade

Before analyzing networks, consider two queues in series: jobs departing Queue 1 immediately enter Queue 2.
For Queue 2 to be tractable via classical Markovian queueing theory ($M/M/1$), its arrival process must be Poisson. But does an $M/M/1$ queue output a Poisson process?

### Burke's Theorem (Paul J. Burke, 1956)

**Theorem**: Consider an $M/M/c$ queue in steady state with arrival rate $\lambda$ and service rate $\mu$ per server ($\rho = \lambda / (c\mu) < 1$).
1. The departure process is a **Poisson process** with the exact same rate $\lambda$ as the arrival process.
2. At any arbitrary time $t$, the number of jobs currently in the system $N(t)$ is **statistically independent** of the sequence of departure times prior to $t$.

### Reversibility and the Physics of Burke's Law
Burke's theorem can be proven via Kolmogorov's criterion of **time reversibility**.
If you record the birth-death Markov chain of an $M/M/1$ queue in steady state and play the film backward in time:
- In reversed time, departures look like arrivals (which are Poisson with rate $\lambda$).
- The reversed process has the exact same stationary distribution as the forward process.
- Therefore, in forward time, departures must be a Poisson process with rate $\lambda$!

**Engineering Consequence**:
In an open feed-forward pipeline of $M/M/c$ queues without feedback, each downstream stage receives a pure Poisson arrival stream. Each stage can be solved completely independently as an isolated $M/M/c$ queue!

---

## 3. Jackson Open Queueing Networks

What happens when there are feedback loops—jobs returning to earlier stations after failing a check?
In the presence of feedback, the inter-arrival stream at a queue is no longer Poisson (it is bursty and auto-correlated). Remarkably, James R. Jackson (1957) proved that the system still decomposes into independent product-form marginals!

### Network Specification:
Consider an open network of $K$ service stations:
- Station $i \in \{1, \dots, K\}$ has $c_i$ servers, each operating at exponential service rate $\mu_i$.
- External arrivals enter station $i$ from the outside world according to an independent Poisson process with rate $\gamma_i \ge 0$.
- When a job completes service at station $i$, it is routed to station $j$ with probability $r_{ij}$, or exits the network with probability:

$$
r_{i0} = 1 - \sum_{j=1}^K r_{ij} \ge 0
$$

- The matrix $P = (r_{ij})_{K \times K}$ is the **routing transition matrix**.

### The Traffic Equations (Flow Balance)

In steady state, the total arrival rate $\lambda_i$ at station $i$ must equal the sum of external arrivals plus internal transfers from all stations $j$:

$$
\lambda_i = \gamma_i + \sum_{j=1}^K \lambda_j r_{ji}, \quad \text{for } i = 1, \dots, K
$$

In vector-matrix form:

$$
\boldsymbol{\lambda} = \boldsymbol{\gamma} + \boldsymbol{\lambda} P
$$

Rearranging:

$$
\boldsymbol{\lambda}(I - P) = \boldsymbol{\gamma}
$$

If the network is open (every job eventually leaves, so $(I - P)$ is invertible):

$$
\boldsymbol{\lambda} = \boldsymbol{\gamma} (I - P)^{-1}
$$

The matrix $M = (I - P)^{-1} = \sum_{k=0}^\infty P^k$ is the **fundamental matrix** of the network. Entry $M_{ji}$ represents the average number of visits a job entering at station $j$ will make to station $i$ before departing the network!

### Stability Condition:
For the network to reach steady state, every station must operate below capacity:

$$
\rho_i = \frac{\lambda_i}{c_i \mu_i} < 1, \quad \forall i \in \{1, \dots, K\}
$$

---

## 4. Jackson's Product-Form Theorem (1957)

**Theorem**: If $\rho_i < 1$ for all stations $i$, the joint equilibrium probability distribution of the queue lengths $(N_1, N_2, \dots, N_K)$ factors into the product of independent marginal distributions:

$$
P(n_1, n_2, \dots, n_K) = \prod_{i=1}^K P_i(n_i)
$$

For single-server stations ($c_i = 1$):

$$
P(n_1, n_2, \dots, n_K) = \prod_{i=1}^K (1 - \rho_i) \rho_i^{n_i}
$$

### The Jackson breakthrough:
Despite feedback loops causing non-Poisson arrival streams at individual stations, **in equilibrium, each station behaves exactly as if it were an independent $M/M/1$ queue with arrival rate $\lambda_i$!**

### Network-Wide Metrics via Little's Law:

1. **Expected number of jobs at station $i$**:

$$
\mathbb{E}[N_i] = \frac{\rho_i}{1 - \rho_i}
$$

2. **Total number of jobs in the entire network**:

$$
\mathbb{E}[N] = \sum_{i=1}^K \mathbb{E}[N_i] = \sum_{i=1}^K \frac{\rho_i}{1 - \rho_i}
$$

3. **Total external arrival rate**:

$$
\Gamma = \sum_{i=1}^K \gamma_i
$$

4. **Mean End-to-End System Residency Time**:
   By Little's Law applied to the entire network:

$$
\mathbb{E}[T_{\text{net}}] = \frac{\mathbb{E}[N]}{\Gamma} = \frac{1}{\Gamma} \sum_{i=1}^K \frac{\rho_i}{1 - \rho_i}
$$

   Equivalently, by linearity of expectation:

$$
\mathbb{E}[T_{\text{net}}] = \sum_{i=1}^K V_i \mathbb{E}[T_i] = \sum_{i=1}^K \left(\frac{\lambda_i}{\Gamma}\right) \frac{1}{\mu_i(1 - \rho_i)}
$$

   where $V_i = \lambda_i / \Gamma$ is the average number of visits per job to station $i$.

---

## 5. Bottleneck Analysis & The Amdahl's Law of Queueing

In any multi-stage queueing network, the **bottleneck station** $i^*$ is the station with the highest utilization:

$$
i^* = \arg\max_{i} \rho_i = \arg\max_i \left(\frac{\lambda_i}{c_i \mu_i}\right)
$$

### Asymptotic Fleet Scaling Laws:
1. **Maximum Throughput**: The maximum external workload the network can support before exploding is limited strictly by the bottleneck:

$$
\Gamma_{\max} = \frac{\Gamma}{\rho_{i^*}}
$$

2. **The Amdahl Trap**: Optimizing a non-bottleneck stage (e.g. speeding up linting from 2 seconds to 0.5 seconds when compilation is at 94% utilization) yields:
   - **Zero** increase in total network throughput capacity,
   - **Virtually zero** reduction in end-to-end developer turnaround time.
3. **Targeted Capital Allocation**: Every dollar of compute capacity or optimization effort should be directed exclusively to the bottleneck station $i^*$ until another station becomes the new bottleneck!

---

## 6. Load-Balanced Worker Pools: The Power-of-Two-Choices

Now consider routing incoming jobs to a pool of $N$ identical parallel servers:

$$
\text{Incoming Jobs } \lambda \longrightarrow \text{Load Balancer} \longrightarrow \{ \text{Server } 1, \dots, \text{Server } N \}
$$

How should the load balancer assign jobs to servers?

### Strategy 1: Random Routing (Round-Robin / Uniform Hash)
Each job is sent to a server chosen uniformly at random.
- Each server behaves as an independent $M/M/1$ queue with rate $\lambda / N$.
- Tail distribution: Probability that a server has $\ge i$ jobs decays purely exponentially:

$$
P(N_{\text{server}} \ge i) = \rho^i
$$

- Expected maximum queue length across $N$ servers:

$$
\mathbb{E}[\max_{k} N_k] \approx \frac{\log N}{\log(1/\rho)}
$$

  As fleet size $N$ grows, the worst-case queue length grows logarithmically.

### Strategy 2: Join-the-Shortest-Queue (JSQ)
Query all $N$ servers and send the job to the server with the absolute shortest queue.
- Optimal, but requires $O(N)$ communication overhead per job arrival. In large distributed fleets ($N = 10,000$), central state tracking creates a severe coordinator bottleneck.

### Strategy 3: The Power-of-Two-Choices (Mitzenmacher 1996, Vvedenskaya et al. 1996)
Instead of querying all $N$ servers, **sample $d = 2$ servers uniformly at random**, and route the job to whichever of the two has the shorter queue!

**Theorem (Supermarket Model)**:
Under the Power-of-Two-Choices rule ($d = 2$), as $N \to \infty$, the fraction of queues with $\ge i$ jobs decays **doubly exponentially**:

$$
P(N_{\text{server}} \ge i) \approx \rho^{\frac{2^i - 1}{2 - 1}} = \rho^{2^i - 1}
$$

Comparing the tail decay for $\rho = 0.90$:
- **Random ($d=1$)**:
  - $i=1$: $0.90^1 = 0.900$
  - $i=2$: $0.90^2 = 0.810$
  - $i=3$: $0.90^3 = 0.729$
  - $i=4$: $0.90^4 = 0.656$
  - $i=5$: $0.90^5 = 0.590$
- **Power-of-Two ($d=2$)**:
  - $i=1$: $0.90^1 = 0.900$
  - $i=2$: $0.90^3 = 0.729$
  - $i=3$: $0.90^7 = 0.478$
  - $i=4$: $0.90^{15} = 0.206$
  - $i=5$: $0.90^{31} = 0.038$
  - $i=6$: $0.90^{63} = 0.0013$!

### Maximum Queue Length Collapse:

$$
\mathbb{E}[\max_{k} N_k] = \frac{\log \log N}{\log 2} + O(1)
$$

Moving from $d=1$ to $d=2$ achieves an exponential improvement in tail queue length with only **2 probe requests per arrival**, providing 99% of the benefits of full JSQ at virtually zero coordinator overhead!

---

## 7. Concrete Step-by-Step Worked Example with Real Numbers

Let us analyze an end-to-end multi-stage Continuous Integration pipeline consisting of 4 interconnected service tiers:
1. **Station 1 (Static Analysis & Linting)**
2. **Station 2 (Compilation & Build)**
3. **Station 3 (Unit Test Suite)**
4. **Station 4 (End-to-End Integration & Device Test)**

### Pipeline Flow Dynamics:
- External developer pull requests arrive at Station 1 with rate $\gamma_1 = 40\text{ PRs/hour}$. No external arrivals at other stations ($\gamma_2 = \gamma_3 = \gamma_4 = 0$).
- **Station 1 (Linting)**:
  - 100% of successful jobs pass to Station 2 ($r_{12} = 1.0$).
  - Service rate: $\mu_1 = 120\text{ PRs/hour}$. Single-server pool ($c_1 = 1$).
- **Station 2 (Build)**:
  - 95% of jobs compile successfully and proceed to Unit Testing ($r_{23} = 0.95$).
  - 5% fail compile and exit the pipeline ($r_{20} = 0.05$).
  - Service rate: $\mu_2 = 60\text{ PRs/hour}$. Single-server pool ($c_2 = 1$).
- **Station 3 (Unit Test)**:
  - 85% of unit tests pass and proceed to Integration Testing ($r_{34} = 0.85$).
  - 10% fail permanently and exit ($r_{30} = 0.10$).
  - **Feedback Loop**: 5% encounter flaky compilation cache corruption and loop back to Station 2 for a clean re-build ($r_{32} = 0.05$).
  - Service rate: $\mu_3 = 50\text{ PRs/hour}$. Single-server pool ($c_3 = 1$).
- **Station 4 (E2E Integration Test)**:
  - 90% pass and complete verification ($r_{40} = 0.90$).
  - 10% fail integration and trigger a targeted diagnostic re-run back at Station 3 ($r_{43} = 0.10$).
  - Service rate: $\mu_4 = 40\text{ PRs/hour}$. Single-server pool ($c_4 = 1$).

---

### Step 1: Solve the Traffic Flow Equations

The routing matrix $P$ is:

$$
P = \begin{pmatrix}
0 & 1.0 & 0 & 0 \\
0 & 0 & 0.95 & 0 \\
0 & 0.05 & 0 & 0.85 \\
0 & 0 & 0.10 & 0
\end{pmatrix}
$$

The traffic equations $\lambda_i = \gamma_i + \sum_j \lambda_j r_{ji}$:
1. $\lambda_1 = 40$
2. $\lambda_2 = \lambda_1 (1.0) + \lambda_3 (0.05) = 40 + 0.05 \lambda_3$
3. $\lambda_3 = \lambda_2 (0.95) + \lambda_4 (0.10)$
4. $\lambda_4 = \lambda_3 (0.85)$

Substituting Equation 4 into Equation 3:

$$
\lambda_3 = 0.95 \lambda_2 + 0.10 (0.85 \lambda_3) = 0.95 \lambda_2 + 0.085 \lambda_3
$$

$$
(1 - 0.085) \lambda_3 = 0.95 \lambda_2 \implies 0.915 \lambda_3 = 0.95 \lambda_2 \implies \lambda_3 = \frac{0.95}{0.915} \lambda_2 \approx 1.03825 \lambda_2

$$
Substitute into Equation 2:
$$

\lambda_2 = 40 + 0.05 (1.03825 \lambda_2) = 40 + 0.05191 \lambda_2

$$

$$

(1 - 0.05191) \lambda_2 = 40 \implies 0.94809 \lambda_2 = 40 \implies \boldsymbol{\lambda_2 = 42.190\text{ PRs/hour}}

$$
Now calculate the remaining rates:
- $\lambda_1 = \mathbf{40.000\text{ PRs/hour}}$
- $\lambda_3 = 1.03825 \times 42.190 = \mathbf{43.804\text{ PRs/hour}}$
- $\lambda_4 = 0.85 \times 43.804 = \mathbf{37.233\text{ PRs/hour}}$

Notice how feedback loops amplified the arrival rates at Stations 2 and 3 above the initial $40\text{ PRs/hr}$!

---

### Step 2: Compute Station Utilizations and Identify the Bottleneck

With service rates $\mu = (120, 60, 50, 40)$:
- **Station 1 (Linting)**:
$$

  \rho_1 = \frac{\lambda_1}{\mu_1} = \frac{40.0}{120.0} = \mathbf{0.3333} \quad (33.3\%)

$$
- **Station 2 (Build)**:
$$

  \rho_2 = \frac{\lambda_2}{\mu_2} = \frac{42.190}{60.0} = \mathbf{0.7032} \quad (70.3\%)

$$
- **Station 3 (Unit Test)**:
$$

  \rho_3 = \frac{\lambda_3}{\mu_3} = \frac{43.804}{50.0} = \mathbf{0.8761} \quad (87.6\%)

$$
- **Station 4 (E2E Integration)**:
$$

  \rho_4 = \frac{\lambda_4}{\mu_4} = \frac{37.233}{40.0} = \mathbf{0.9308} \quad (\mathbf{93.1\%})

$$
### Bottleneck Identification:
$$

i^* = \arg\max \rho_i = \mathbf{\text{Station 4 (E2E Integration)}} \quad (\rho_4 = 93.1\%)

$$
Station 4 is the critical bottleneck pushing into the heavy-traffic danger zone!

---

### Step 3: Compute Expected Queue Lengths and End-to-End Latency

Using Jackson's Theorem, each station behaves as an independent $M/M/1$ queue:
$$

\mathbb{E}[N_i] = \frac{\rho_i}{1 - \rho_i}

$$
- Station 1: $\mathbb{E}[N_1] = \frac{0.3333}{0.6667} = 0.500\text{ PRs}$
- Station 2: $\mathbb{E}[N_2] = \frac{0.7032}{0.2968} = 2.369\text{ PRs}$
- Station 3: $\mathbb{E}[N_3] = \frac{0.8761}{0.1239} = 7.071\text{ PRs}$
- Station 4: $\mathbb{E}[N_4] = \frac{0.9308}{0.0692} = 13.451\text{ PRs}$

Total PRs in pipeline across all stages:
$$

\mathbb{E}[N] = 0.500 + 2.369 + 7.071 + 13.451 = \mathbf{23.391\text{ PRs}}

$$
Notice: The bottleneck Station 4 holds **$57.5\%$ of all in-flight PRs**!

### End-to-End Turnaround Time:
By Little's Law with total external arrival rate $\Gamma = 40\text{ PRs/hour}$:
$$

\mathbb{E}[T_{\text{net}}] = \frac{\mathbb{E}[N]}{\Gamma} = \frac{23.391}{40} = 0.5848\text{ hours} = \mathbf{35.09\text{ minutes}}

$$
### Step 4: The Impact of Bottleneck Relief

Suppose we add a second worker to the bottleneck Station 4, doubling its capacity to $\mu_4 = 80\text{ PRs/hr}$ ($c_4 = 2$ or 2 parallel runners):
New Station 4 utilization:
$$

\rho_4^{\text{new}} = \frac{37.233}{80.0} = 0.4654

$$
Expected jobs at Station 4 drops from $13.451$ down to:
$$

\mathbb{E}[N_4^{\text{new}}] \approx \frac{0.4654}{1 - 0.4654} \approx 0.870\text{ PRs}

$$
New total pipeline backlog:
$$

\mathbb{E}[N^{\text{new}}] = 0.500 + 2.369 + 7.071 + 0.870 = 10.810\text{ PRs}

$$
New turnaround time:
$$

\mathbb{E}[T_{\text{net}}^{\text{new}}] = \frac{10.810}{40} = 0.2703\text{ hours} = \mathbf{16.22\text{ minutes}}
$$
Doubling capacity at the single bottleneck station reduced total developer wait time across the entire 4-stage pipeline by **$53.8\%$**!
